In [1]:
import os

import pandas as pd
from tqdm import tqdm

%cd '/pfs/work9/workspace/scratch/ma_faroesch-master-thesis/playground-uc3/master-thesis/'

/pfs/work9/workspace/scratch/ma_faroesch-master-thesis/playground-uc3/master-thesis


In [2]:
def create_summary_table(df: pd.DataFrame, filter_tag:list[str], reference_measure:str) -> pd.DataFrame:
    """Erstellt eine Tabelle mit den Bestwerten pro Run."""
    print("\nErstelle Summary Tabelle (Max-Werte)...")

    # filter df for each column where df[column] is true
    for tag in filter_tag:
        col = f"tag_{tag}"
        if col not in df.columns:
            raise ValueError(f"Tag-Spalte {col} existiert nicht im DataFrame.")
        df = df[df[col] == True]

    if df.empty:
        print("DataFrame nach Tag-Filter leer.")
        return df

    # Sicherstellen, dass innerhalb jedes Runs nach step sortiert ist
    df = df.sort_values(["run_id", "_step"]).copy()

    # Epoch als laufende Evaluation pro run_id (startet bei 0)
    df["epoch"] = df.groupby("run_id").cumcount()

    # Bestes F1-Concept pro Run auswählen
    best_per_run = (
        df.loc[df.groupby("run_id")[reference_measure].idxmax()]
        .reset_index(drop=True)
    )
    
    return best_per_run

In [5]:
DATA_PATH = 'thesis-figures/epg_cbm/results/1_epg_cbm_results.parquet'

data = pd.read_parquet(DATA_PATH)
val_results_baseline = create_summary_table(df=data, filter_tag=['baseline'], reference_measure='F1 Labels')


Erstelle Summary Tabelle (Max-Werte)...


In [6]:
val_results_baseline

,_step,Loss EPG,F1 Concepts,F1 Labels,Runtime,Epoch,run_id,Attribution Method,$\lambda_\mathrm{EPG}$,EPG-level,config_label,Dataset,tag_result,tag_baseline,epoch
0,10933,0.0,0.991027,1.000000,4323.008494,14,1x0ikimu,NaN,1.0,None,λ=1.0,FunnyBirds,True,True,13
1,2397,0.0,0.475560,0.688527,14520.743311,32,7h64wol2,NaN,1.0,None,λ=1.0,CUB_112,True,True,31


In [ ]:
data = pd.read_parquet(DATA_PATH)
data['EPG-Score'] = 1 - data['Loss EPG']

data = data[(data["tag_result"] == True) & (data['Attribution Method'].notna())].copy()

In [49]:
val_results = create_summary_table(df=data, filter_tag=['result'], reference_measure='F1 Labels')


Erstelle Summary Tabelle (Max-Werte)...


In [50]:
val_results['Dataset'].value_counts()

Dataset
CUB_112       42
FunnyBirds    42
Name: count, dtype: int64

In [51]:
val_results.columns

Index(['_step', 'Loss EPG', 'F1 Concepts', 'F1 Labels', 'Runtime', 'Epoch',
       'run_id', 'Attribution Method', '$\lambda_\mathrm{EPG}$', 'EPG-level',
       'config_label', 'Dataset', 'tag_result', 'tag_baseline', 'EPG-Score',
       'epoch'],
      dtype='object')

In [52]:
val_results_paper = val_results[['run_id', 'Dataset', 'Attribution Method', r"$\lambda_\mathrm{EPG}$", 'EPG-level', 'EPG-Score', 'F1 Concepts', 'F1 Labels']].copy()

In [53]:
val_results_paper['Dataset']

0        CUB_112
1     FunnyBirds
2     FunnyBirds
3     FunnyBirds
4     FunnyBirds
         ...    
79       CUB_112
80    FunnyBirds
81    FunnyBirds
82       CUB_112
83    FunnyBirds
Name: Dataset, Length: 84, dtype: object

# FunnyBirds

In [54]:
metrics = ['EPG-Score', 'F1 Concepts', 'F1 Labels']

df_wide = (
    val_results_paper[val_results_paper['Dataset']=='FunnyBirds']
        # .sort_values(by=["Dataset", "Attribution Method"])
        .pivot(
            index=['Attribution Method', r"$\lambda_\mathrm{EPG}$"],
            columns='EPG-level',
            values=metrics
        )
)

In [60]:
df_wide

EPG-Score            \
EPG-level                                         concept     image   
Attribution Method       $\lambda_\mathrm{EPG}$                       
GradCAM                  0.001                   0.327724  0.380383   
                         0.005                   0.343623  0.334878   
                         0.010                   0.375977  0.473571   
                         0.050                   0.440929  0.619234   
                         0.100                   0.503903  0.687864   
                         0.500                   0.551297  0.792878   
                         1.000                   0.583582  0.793639   
GradCAM++                0.001                   0.473587  0.469259   
                         0.005                   0.472453  0.483816   
                         0.010                   0.480429  0.489896   
                         0.050                   0.501722  0.590062   
                         0.100                   0.517100  0.652152   
                         0.500                   0.603331  0.685513   
                         1.000                   0.686515  0.753059   
Input $\times$ Gradients 0.001                   0.363343  0.380618   
                         0.005                   0.378055  0.377794   
                         0.010                   0.373348  0.375513   
                         0.050                   0.371055  0.374093   
                         0.100                   0.375092  0.402122   
                         0.500                   0.417011  0.432140   
                         1.000                   0.419061  0.448798   

                                                F1 Concepts            \
EPG-level                                           concept     image   
Attribution Method       $\lambda_\mathrm{EPG}$                         
GradCAM                  0.001                     0.380971  0.484410   
                         0.005                     0.405044  0.383641   
                         0.010                     0.472971  0.496836   
                         0.050                     0.489046  0.490209   
                         0.100                     0.488725  0.497167   
                         0.500                     0.464016  0.488308   
                         1.000                     0.467843  0.485519   
GradCAM++                0.001                     0.476991  0.489760   
                         0.005                     0.494960  0.443205   
                         0.010                     0.443127  0.468422   
                         0.050                     0.500262  0.499300   
                         0.100                     0.482849  0.498398   
                         0.500                     0.451300  0.359988   
                         1.000                     0.467198  0.368302   
Input $\times$ Gradients 0.001                     0.368136  0.407010   
                         0.005                     0.367138  0.423036   
                         0.010                     0.380253  0.379909   
                         0.050                     0.343136  0.399026   
                         0.100                     0.435624  0.407577   
                         0.500                     0.427600  0.335651   
                         1.000                     0.380598  0.323906   

                                                F1 Labels            
EPG-level                                         concept     image  
Attribution Method       $\lambda_\mathrm{EPG}$                      
GradCAM                  0.001                   0.700211  0.696924  
                         0.005                   0.699630  0.697628  
                         0.010                   0.714100  0.696217  
                         0.050                   0.704276  0.701152  
                         0.100                   0.694462  0.694454  
                         0.500

In [61]:
code = df_wide.to_latex(float_format="%.4f")
print(code)

\begin{tabular}{llrrrrrr}
\toprule
 &  & \multicolumn{2}{r}{EPG-Score} & \multicolumn{2}{r}{F1 Concepts} & \multicolumn{2}{r}{F1 Labels} \\
 & EPG-level & concept & image & concept & image & concept & image \\
Attribution Method & $\lambda_\mathrm{EPG}$ &  &  &  &  &  &  \\
\midrule
\multirow[t]{7}{*}{GradCAM} & 0.001000 & 0.3277 & 0.3804 & 0.3810 & 0.4844 & 0.7002 & 0.6969 \\
 & 0.005000 & 0.3436 & 0.3349 & 0.4050 & 0.3836 & 0.6996 & 0.6976 \\
 & 0.010000 & 0.3760 & 0.4736 & 0.4730 & 0.4968 & 0.7141 & 0.6962 \\
 & 0.050000 & 0.4409 & 0.6192 & 0.4890 & 0.4902 & 0.7043 & 0.7012 \\
 & 0.100000 & 0.5039 & 0.6879 & 0.4887 & 0.4972 & 0.6945 & 0.6945 \\
 & 0.500000 & 0.5513 & 0.7929 & 0.4640 & 0.4883 & 0.6909 & 0.6700 \\
 & 1.000000 & 0.5836 & 0.7936 & 0.4678 & 0.4855 & 0.6914 & 0.6655 \\
\cline{1-8}
\multirow[t]{7}{*}{GradCAM++} & 0.001000 & 0.4736 & 0.4693 & 0.4770 & 0.4898 & 0.6895 & 0.6995 \\
 & 0.005000 & 0.4725 & 0.4838 & 0.4950 & 0.4432 & 0.7036 & 0.7005 \\
 & 0.010000 & 0.4804 & 0.48

# CUB-112

In [62]:
metrics = ['EPG-Score', 'F1 Concepts', 'F1 Labels']

df_wide = (
    val_results_paper[val_results_paper['Dataset']=='CUB_112']
        # .sort_values(by=["Dataset", "Attribution Method"])
        .pivot(
            index=['Attribution Method', r"$\lambda_\mathrm{EPG}$"],
            columns='EPG-level',
            values=metrics
        )
)

In [63]:
df_wide

EPG-Score            \
EPG-level                                         concept     image   
Attribution Method       $\lambda_\mathrm{EPG}$                       
GradCAM                  0.001                   0.327724  0.380383   
                         0.005                   0.343623  0.334878   
                         0.010                   0.375977  0.473571   
                         0.050                   0.440929  0.619234   
                         0.100                   0.503903  0.687864   
                         0.500                   0.551297  0.792878   
                         1.000                   0.583582  0.793639   
GradCAM++                0.001                   0.473587  0.469259   
                         0.005                   0.472453  0.483816   
                         0.010                   0.480429  0.489896   
                         0.050                   0.501722  0.590062   
                         0.100                   0.517100  0.652152   
                         0.500                   0.603331  0.685513   
                         1.000                   0.686515  0.753059   
Input $\times$ Gradients 0.001                   0.363343  0.380618   
                         0.005                   0.378055  0.377794   
                         0.010                   0.373348  0.375513   
                         0.050                   0.371055  0.374093   
                         0.100                   0.375092  0.402122   
                         0.500                   0.417011  0.432140   
                         1.000                   0.419061  0.448798   

                                                F1 Concepts            \
EPG-level                                           concept     image   
Attribution Method       $\lambda_\mathrm{EPG}$                         
GradCAM                  0.001                     0.380971  0.484410   
                         0.005                     0.405044  0.383641   
                         0.010                     0.472971  0.496836   
                         0.050                     0.489046  0.490209   
                         0.100                     0.488725  0.497167   
                         0.500                     0.464016  0.488308   
                         1.000                     0.467843  0.485519   
GradCAM++                0.001                     0.476991  0.489760   
                         0.005                     0.494960  0.443205   
                         0.010                     0.443127  0.468422   
                         0.050                     0.500262  0.499300   
                         0.100                     0.482849  0.498398   
                         0.500                     0.451300  0.359988   
                         1.000                     0.467198  0.368302   
Input $\times$ Gradients 0.001                     0.368136  0.407010   
                         0.005                     0.367138  0.423036   
                         0.010                     0.380253  0.379909   
                         0.050                     0.343136  0.399026   
                         0.100                     0.435624  0.407577   
                         0.500                     0.427600  0.335651   
                         1.000                     0.380598  0.323906   

                                                F1 Labels            
EPG-level                                         concept     image  
Attribution Method       $\lambda_\mathrm{EPG}$                      
GradCAM                  0.001                   0.700211  0.696924  
                         0.005                   0.699630  0.697628  
                         0.010                   0.714100  0.696217  
                         0.050                   0.704276  0.701152  
                         0.100                   0.694462  0.694454  
                         0.500

In [64]:
code = df_wide.to_latex(float_format="%.4f")
print(code)

\begin{tabular}{llrrrrrr}
\toprule
 &  & \multicolumn{2}{r}{EPG-Score} & \multicolumn{2}{r}{F1 Concepts} & \multicolumn{2}{r}{F1 Labels} \\
 & EPG-level & concept & image & concept & image & concept & image \\
Attribution Method & $\lambda_\mathrm{EPG}$ &  &  &  &  &  &  \\
\midrule
\multirow[t]{7}{*}{GradCAM} & 0.001000 & 0.3277 & 0.3804 & 0.3810 & 0.4844 & 0.7002 & 0.6969 \\
 & 0.005000 & 0.3436 & 0.3349 & 0.4050 & 0.3836 & 0.6996 & 0.6976 \\
 & 0.010000 & 0.3760 & 0.4736 & 0.4730 & 0.4968 & 0.7141 & 0.6962 \\
 & 0.050000 & 0.4409 & 0.6192 & 0.4890 & 0.4902 & 0.7043 & 0.7012 \\
 & 0.100000 & 0.5039 & 0.6879 & 0.4887 & 0.4972 & 0.6945 & 0.6945 \\
 & 0.500000 & 0.5513 & 0.7929 & 0.4640 & 0.4883 & 0.6909 & 0.6700 \\
 & 1.000000 & 0.5836 & 0.7936 & 0.4678 & 0.4855 & 0.6914 & 0.6655 \\
\cline{1-8}
\multirow[t]{7}{*}{GradCAM++} & 0.001000 & 0.4736 & 0.4693 & 0.4770 & 0.4898 & 0.6895 & 0.6995 \\
 & 0.005000 & 0.4725 & 0.4838 & 0.4950 & 0.4432 & 0.7036 & 0.7005 \\
 & 0.010000 & 0.4804 & 0.48